In [ ]:
import numpy as np
import cvxpy as cp
import pandas as pd
from pathlib import Path

In [ ]:
cp.MAX_NUM_SUBEXPRESSIONS = 10000

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data/combined_data_positive_gen.csv"
df = pd.read_csv(DATA_DIR, 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  
                     index_col='ts')

n = df["Collective PV"].size

In [ ]:
## Defining a constant 2 kW load
Pload_np = np.zeros((n))+1 
Pload = cp.Constant(Pload_np)
dT = 0.25 #h

## Target min self-sufficiency
minimum_self_sufficiency = 0.8

## Battery parameters; power and energy capacities are different values :) -- efficiency refers to power, always
charge_eff = 0.95
discharge_eff = 0.95
P_capacity = cp.Variable()
E_capacity = cp.Variable()

## Defining generation values from Syslab Data 
Pgen_pd = df["Collective PV"] + df["Aircon_WT Power"]
Pgen = cp.Constant(Pgen_pd)

## Defining charge and discharge power values
Pcharge = cp.Variable(n,nonneg=True) # charge is positive
Pdischarge = cp.Variable(n,nonneg=True) # discharge is positive

## Defining power imported and exported to the grid -- becomes 0 with 100% self-sufficiency
Pgrid_buy = cp.Variable(n, nonneg=True)
Pgrid_sell = cp.Variable(n, nonneg=True)

## Defining variable SOC
SOC = cp.Variable(n)

## Defining equation for self-sufficiency 
self_sufficiency = (Pgen - Pcharge - Pgrid_sell)/Pload 
average_self_sufficiency = sum(self_sufficiency)/n

In [ ]:
constraints = [Pcharge >= 0,
               Pdischarge >= 0,
               Pcharge <= P_capacity,
               Pdischarge <= P_capacity]
constraints += [SOC >= E_capacity * 0.1, SOC <= E_capacity]
constraints += [Pgen + Pdischarge - Pcharge + Pgrid_buy - Pgrid_sell == Pload, # power flow equation 
               SOC[0] == 0.5 * E_capacity + (Pcharge[0]*charge_eff - Pdischarge[0]/ discharge_eff)*dT, ## Starting with 50% charge
               SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff)*dT, # battery state definition
               SOC[n-1] >= 0.5 * E_capacity, # minimum 10% discharge
               average_self_sufficiency >= minimum_self_sufficiency # average self sufficiency over the year
               ]
               

In [ ]:
problem2 = cp.Problem(cp.Minimize(E_capacity), constraints)
problem2.solve(verbose=True)

target function could be to minimize SOC * battery E capacity * 1.0 or just the E_capacity
you need to make the 